Mathematical Precision


In [1]:
import torch
import torch.nn as nn

# A simplified version of an EDSR Residual Block
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1)
        )

    def forward(self, x):
        return x + self.body(x) * 0.1 # Residual scaling for stability

class EDSRPrecision(nn.Module):
    def __init__(self, upscale_factor=4, channels=3):
        super().__init__()
        self.head = nn.Conv2d(channels, 64, 3, padding=1)
        
        # Stack of Residual Blocks (Precision core)
        self.body = nn.Sequential(*[ResBlock(64) for _ in range(16)])
        
        self.upsampler = nn.Sequential(
            nn.Conv2d(64, 64 * (upscale_factor ** 2), 3, padding=1),
            nn.PixelShuffle(upscale_factor),
            nn.Conv2d(64, channels, 3, padding=1)
        )

    def forward(self, x):
        x = self.head(x)
        res = self.body(x)
        x = x + res
        return self.upsampler(x)

# Usage
model_precision = EDSRPrecision(upscale_factor=4)
# print(model_precision)

ModuleNotFoundError: No module named 'torch'

In [3]:
import torch
import torch.nn as nn

class HSIPrecisionNet(nn.Module):
    def __init__(self, in_channels, scale_factor=4):
        super(HSIPrecisionNet, self). __init__()
        
        # 3D Convolutions preserve the relationship between neighboring spectral bands
        self.feature_extractor = nn.Sequential(
            nn.Conv3d(1, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.ReLU(inplace=True),
            nn.Conv3d(64, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.ReLU(inplace=True)
        )
        
        # Spatial Upsampling
        self.upsampler = nn.Sequential(
            nn.Conv2d(64 * in_channels, 64 * (scale_factor**2), kernel_size=3, padding=1),
            nn.PixelShuffle(scale_factor),
            nn.Conv2d(64, in_channels, kernel_size=3, padding=1)
        )

    def forward(self, x):
        # x shape: (Batch, Channels, H, W)
        b, c, h, w = x.shape
        x = x.unsqueeze(1) # Add a dimension for 3D Conv: (B, 1, C, H, W)
        
        feat = self.feature_extractor(x) # (B, 64, C, H, W)
        
        # Flatten spectral dim into channels for the 2D upsampler
        feat = feat.view(b, -1, h, w) 
        out = self.upsampler(feat)
        return out

# Example: HSI with 31 spectral bands
model = HSIPrecisionNet(in_channels=31, scale_factor=4)

Visual beaaty

In [4]:
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet
import cv2

def enhance_visual_beauty(image_path, output_path):
    # 1. Define the Architecture (RRDBNet is the backbone of ESRGAN)
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
    
    # 2. Set up the Upsampler (Downloads pre-trained "beauty" weights automatically)
    upsampler = RealESRGANer(
        scale=4,
        model_path='https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
        model=model,
        tile=0,
        tile_pad=10,
        pre_pad=0,
        half=True # Use FP16 for speed if you have a GPU
    )

    img = cv2.imread(image_path)
    output, _ = upsampler.enhance(img, outscale=4)
    
    cv2.imwrite(output_path, output)
    print(f"Enhanced image saved to {output_path}")

# enhance_visual_beauty('input.jpg', 'enhanced.jpg')

In [1]:
import torch
import torch.nn as nn

class SpatialSpectralAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        # Spectral Attention: Which bands matter most?
        self.spectral_attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // 4, 1),
            nn.ReLU(),
            nn.Conv2d(channels // 4, channels, 1),
            nn.Sigmoid()
        )
        # Spatial Attention: Where are the sharp edges?
        self.spatial_attn = nn.Sequential(
            nn.Conv2d(channels, 1, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x * self.spectral_attn(x)
        x = x * self.spatial_attn(x)
        return x

class HSIBeautyNet(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.head = nn.Conv2d(in_channels, 64, 3, padding=1)
        self.attention = SpatialSpectralAttention(64)
        self.tail = nn.Conv2d(64, in_channels * 16, 3, padding=1)
        self.pixel_shuffle = nn.PixelShuffle(4)

    def forward(self, x):
        res = self.head(x)
        res = self.attention(res)
        out = self.pixel_shuffle(self.tail(res))
        return out

model_beauty = HSIBeautyNet(in_channels=31)

Dataset Loader

In [2]:
import torch
import scipy.io as sio
import numpy as np
from torch.utils.data import Dataset, DataLoader

class HSIDataLoader(Dataset):
    def __init__(self, file_path, data_key, scale=4, patch_size=32):
        """
        file_path: Path to the .mat file
        data_key: The dictionary key for the data in the .mat file
        """
        # Load .mat file
        mat_data = sio.loadmat(file_path)
        data = mat_data[data_key].astype(np.float32)
        
        # Normalize to 0-1
        data = (data - np.min(data)) / (np.max(data) - np.min(data))
        
        # Convert to Tensor (C, H, W)
        self.hr_data = torch.from_numpy(data).permute(2, 0, 1)
        self.patch_size = patch_size
        self.scale = scale

    def __len__(self):
        # Returns number of patches we can extract
        return (self.hr_data.shape[1] // self.patch_size) * (self.hr_data.shape[2] // self.patch_size)

    def __getitem__(self, idx):
        # Extract a High-Res (HR) patch
        # (For simplicity, this example uses basic indexing)
        c, h, w = self.hr_data.shape
        row = (idx // (w // self.patch_size)) * self.patch_size
        col = (idx % (w // self.patch_size)) * self.patch_size
        
        hr_patch = self.hr_data[:, row:row+self.patch_size, col:col+self.patch_size]
        
        # Create Low-Res (LR) patch by downsampling
        # This simulates the input for your Super-Resolution model
        lr_patch = torch.nn.functional.interpolate(
            hr_patch.unsqueeze(0), 
            scale_factor=1/self.scale, 
            mode='bicubic'
        ).squeeze(0)
        
        return lr_patch, hr_patch

# Example Usage for Different Datasets:
# 1. Indian Pines: data_key='indian_pines_corrected'
# 2. Pavia University: data_key='paviaU'
# 3. Chikusei: data_key='chikusei'

# ds = HSIDataLoader('indian_pines.mat', data_key='indian_pines_corrected')
# loader = DataLoader(ds, batch_size=4, shuffle=True)

In [6]:
import torch
import torch.nn.functional as F

def calculate_psnr(img1, img2):
    """Higher is better (dB)"""
    mse = F.mse_norm(img1, img2) # Mean Squared Error
    if mse == 0: return 100
    return 20 * torch.log10(1.0 / torch.sqrt(mse))

def calculate_sam(img1, img2):
    """Spectral Angle Mapper: Smaller is better (Radians)"""
    # img1, img2 shapes: (C, H, W)
    dot_product = (img1 * img2).sum(dim=0)
    norm_img1 = torch.norm(img1, dim=0)
    norm_img2 = torch.norm(img2, dim=0)
    
    # Cosine similarity clamped for stability
    cos_theta = (dot_product / (norm_img1 * norm_img2 + 1e-8)).clamp(-1, 1)
    sam_val = torch.acos(cos_theta).mean()
    return sam_val

In [2]:
import torch.optim as optim
from tqdm import tqdm

def train_hsi_model(model, dataloader, epochs=50, lr=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.L1Loss() # Better for HSI than MSE

    print(f"Starting Training on {device}...")

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        epoch_psnr = 0
        epoch_sam = 0

        for lr_img, hr_img in tqdm(dataloader, desc=f"Epoch {epoch+1}"):
            lr_img, hr_img = lr_img.to(device), hr_img.to(device)
            
            # Forward pass
            output = model(lr_img)
            loss = criterion(output, hr_img)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Track Metrics
            epoch_loss += loss.item()
            with torch.no_grad():
                epoch_psnr += calculate_psnr(output[0], hr_img[0]).item()
                epoch_sam += calculate_sam(output[0], hr_img[0]).item()

        avg_psnr = epoch_psnr / len(dataloader)
        avg_sam = epoch_sam / len(dataloader)
        
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss/len(dataloader):.4f} | PSNR: {avg_psnr:.2f}dB | SAM: {avg_sam:.4f}")

# Example Run:
# train_hsi_model(hsi_precision_model, indian_pines_loader)

In [5]:
import matplotlib.pyplot as plt
import random

def plot_spectral_signature(hr_img, output_img, pixel_coords=None):
    """
    hr_img: Tensor of shape (C, H, W)
    output_img: Tensor of shape (C, H, W)
    """
    # Move to CPU and convert to numpy
    hr_np = hr_img.detach().cpu().numpy()
    out_np = output_img.detach().cpu().numpy()
    
    num_bands = hr_np.shape[0]
    
    # Pick a random pixel if coordinates aren't provided
    if pixel_coords is None:
        x = random.randint(0, hr_np.shape[1] - 1)
        y = random.randint(0, hr_np.shape[2] - 1)
    else:
        x, y = pixel_coords

    # Extract the spectral vector for that pixel
    hr_signature = hr_np[:, x, y]
    out_signature = out_np[:, x, y]
    
    # Plotting
    plt.figure(figsize=(10, 5))
    plt.plot(range(num_bands), hr_signature, label='Ground Truth (HR)', color='blue', linewidth=2)
    plt.plot(range(num_bands), out_signature, label='Model Output', color='red', linestyle='--', linewidth=2)
    
    plt.title(f"Spectral Signature Comparison at Pixel ({x}, {y})")
    plt.xlabel("Band Number (Wavelength)")
    plt.ylabel("Reflectance / Intensity")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# Example Usage after a forward pass:
# plot_spectral_signature(hr_batch[0], output_batch[0])

In [ ]:
s